Importing the libraries

In [1]:
import pandas as pd
import numpy as np
import re
from gensim.models import KeyedVectors
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize

Defining variables here

In [2]:
TRAIN_DATA = 'train_data.csv'
TEST_DATA = 'test_data.csv'
TITLE_BRAND = 'title_brand.csv'
GOOGLE_VECTORS = 'GoogleNews-vectors-negative300.bin'

Loading the data

In [7]:
train_data = pd.read_csv(TRAIN_DATA)

train_data.head()

/tmp/ipykernel_20208/2754348164.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv(TRAIN_DATA)


,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime
0,2,NaN,False,2016-11-11,A2OSUEZJIN7BI,0511189877,NaN,Chris,I have an older URC-WR7 remote and thought thi...,Cannot Learn,1478822400
1,5,NaN,True,2016-06-06,A2NETQRG6JHIG7,0511189877,NaN,Qrysta White,First time I've EVER had a remote that needed ...,zero programming needed! Miracle!?,1465171200
2,4,NaN,True,2016-03-10,A12JHGROAX49G7,0511189877,NaN,Linwood,Got them and only 2 of them worked. company ca...,Works Good and programs easy.,1457568000
3,5,NaN,True,2016-01-14,A1KV65E2TMMG6F,0511189877,NaN,Dane Williams,I got tired of the remote being on the wrong s...,Same as TWC remote,1452729600
4,5,NaN,True,2016-10-20,A280POPEWI0NSA,0594459451,NaN,Kristina H.,After purchasing cheap cords from another webs...,Good Quality Cord,1476921600


In [8]:
test_data = pd.read_csv(TEST_DATA)

test_data.head()

,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime
0,NaN,True,2018-06-04,A20GGWE66JW9X2,B006Z394GM,{'Color:': ' FPS01-C'},Brian C Toner,The name and description of this device are mi...,The prize for most useless invention of all ti...,1528070400
1,NaN,True,2018-02-18,ARYJWXHEYHG9M,B005BE058W,"{'Size:': ' 1000W', 'Style:': ' G2'}",Snake,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR,1518912000
2,NaN,True,2018-01-20,A10LHZ7WFZ7HLL,B01DA0YCNC,NaN,Amazon Customer,Remote constantly disconnects/ Roku player fre...,Bricked on the regular,1516406400
3,NaN,True,2018-06-25,A11VN8EOHNLP72,B00FBJ4KYC,NaN,Jeremy Bray,I purchased this 4 year protection plan for a ...,DO NOT BUY!!!,1529884800
4,3.0,True,2016-08-17,A194Y8P8TVT7P9,B00P7G82TS,NaN,Mark,I bought one of these and have regretted it ev...,Nightmare - don't buy,1471392000


In [9]:
title_brand = pd.read_csv(TITLE_BRAND)

title_brand.head()

,asin,title,brand
0,0011300000,Genuine Geovision 1 Channel 3rd Party NVR IP S...,GeoVision
1,0043396828,"Books ""Handbook of Astronomical Image Processi...",33 Books Co.
2,0060009810,One Hot Summer,Visit Amazon's Carolina Garcia Aguilera Page
3,0060219602,Hurray for Hattie Rabbit: Story and pictures (...,Visit Amazon's Dick Gackenbach Page
4,0060786817,sex.lies.murder.fame.: A Novel,Visit Amazon's Lolita Files Page


# Section 1: Initial analysis of data

# Section 2: Level of satisfaction with a specific aspect

Let's find similar words to "warranty" and "guarantee"
We'll use pretrained word embeddings.

In [4]:
word_vectors = KeyedVectors.load_word2vec_format(GOOGLE_VECTORS, binary=True)
similar_warranty = [w for w, s in word_vectors.most_similar("warranty", topn=15)]
similar_guarantee = [w for w, s in word_vectors.most_similar("guarantee", topn=15)]
keywords = set(["warranty", "guarantee"] + similar_warranty + similar_guarantee)


In [5]:
print(keywords)

{'guarantees', 'Disclaimer_Past_performance', 'assure', 'warranty', 'assurance', 'Extended_Warranty', 'guarantee', 'Guarantee', 'guarentee', 'guaranteed', 'ensure', 'Warranties', 'guaranteeing', 'assurances', 'assured', 'limited_powertrain_warranty', 'powertrain_warranty', 'Limited_Warranty', 'corrosion_perforation', 'warranties', 'Guaranteed', 'gurantees', 'extended_warranties', 'lifetime_warranty', 'warrantees', 'PMP_MP4_Players', 'warrenty', 'insure', 'Lifetime_Warranty', 'five-year/###_,###_mile', 'Warranty', 'warrantee'}


Filtering reviews containing any of those words

In [10]:
pattern = r'\b(' + '|'.join(re.escape(k) for k in keywords) + r')\b'
train_data['mentions_warranty'] = train_data['reviewText'].str.lower().str.contains(pattern, regex=True)

/tmp/ipykernel_20208/995190863.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  train_data['mentions_warranty'] = train_data['reviewText'].str.lower().str.contains(pattern, regex=True)


In [11]:
train_data.head()

,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,mentions_warranty
0,2,NaN,False,2016-11-11,A2OSUEZJIN7BI,0511189877,NaN,Chris,I have an older URC-WR7 remote and thought thi...,Cannot Learn,1478822400,False
1,5,NaN,True,2016-06-06,A2NETQRG6JHIG7,0511189877,NaN,Qrysta White,First time I've EVER had a remote that needed ...,zero programming needed! Miracle!?,1465171200,False
2,4,NaN,True,2016-03-10,A12JHGROAX49G7,0511189877,NaN,Linwood,Got them and only 2 of them worked. company ca...,Works Good and programs easy.,1457568000,False
3,5,NaN,True,2016-01-14,A1KV65E2TMMG6F,0511189877,NaN,Dane Williams,I got tired of the remote being on the wrong s...,Same as TWC remote,1452729600,False
4,5,NaN,True,2016-10-20,A280POPEWI0NSA,0594459451,NaN,Kristina H.,After purchasing cheap cords from another webs...,Good Quality Cord,1476921600,False


Compute average rating per product

In [13]:
train_data_filtered = train_data[train_data['mentions_warranty']]
result = train_data_filtered.groupby('asin')['overall'].mean().reset_index()
result.rename(columns={'overall': 'mean_overall_warranty'}, inplace=True)


In [15]:
result['count_reviews'] = train_data_filtered.groupby('asin')['overall'].count().values

In [17]:
print(result)

             asin  mean_overall_warranty  count_reviews
0      6541654530                    1.0              1
1      9800466657                    5.0              1
2      B000001OM4                    4.0              1
3      B000001OM5                    5.0              1
4      B000001ON6                    5.0              1
...           ...                    ...            ...
12596  B01HIS5N3K                    4.0              2
12597  B01HIURQWE                    5.0              1
12598  B01HIWBU7Y                    5.0              1
12599  B01HIZEW1C                    5.0              1
12600  B01HJ8E11E                    5.0              1

[12601 rows x 3 columns]


In [18]:
# Create a binary sentiment label from the 'overall' rating:
# - ratings 4 and 5 => positive (1)
# - ratings 1 and 2 => negative (0)
# - rating 3 => neutral (we'll drop these for the binary analysis)
train_data_filtered['sentiment'] = train_data_filtered['overall'].apply(lambda r: 1 if r >= 4 else (0 if r <= 2 else np.nan))
# Drop neutral reviews (overall == 3) for the binary sentiment analysis
train_data_filtered = train_data_filtered.dropna(subset=['sentiment']).copy()
train_data_filtered['sentiment'] = train_data_filtered['sentiment'].astype(int)

# Robust parsing for 'vote' (helpful) field - some rows may be strings like '2 people found this helpful'
import re
def parse_vote(v):
    if pd.isna(v):
        return 0.0
    s = str(v)
    m = re.search(r'\d+', s)
    return float(m.group()) if m else 0.0

train_data_filtered['vote_num'] = train_data_filtered['vote'].apply(parse_vote)

# Normalize 'verified' to a boolean-like flag (handles True/False, 'Y'/'N', 'Yes'/'No', etc.)
train_data_filtered['verified_flag'] = train_data_filtered['verified'].astype(str).str.lower().isin(['true','y','yes','1'])

/tmp/ipykernel_20208/83307303.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data_filtered['sentiment'] = train_data_filtered['overall'].apply(lambda r: 1 if r >= 4 else (0 if r <= 2 else np.nan))


In [19]:
# Aggregate product-level metrics for items that mention warranty/guarantee
agg = train_data_filtered.groupby('asin').agg(
    mean_overall_warranty=('overall', 'mean'),
    count_reviews=('overall', 'count'),
    positive_reviews=('sentiment', lambda x: int((x==1).sum())),
    negative_reviews=('sentiment', lambda x: int((x==0).sum())),
    positivity_rate=('sentiment', lambda x: float(x.mean() if len(x) else np.nan)),
    mean_helpful_votes=('vote_num', 'mean'),
    percent_verified=('verified_flag', 'mean'),
    avg_review_length_words=('reviewText', lambda x: float(x.fillna('').str.split().str.len().mean())),
    avg_summary_length_words=('summary', lambda x: float(x.fillna('').str.split().str.len().mean())),
    first_review_time_unix=('unixReviewTime', 'min'),
    last_review_time_unix=('unixReviewTime', 'max')
)

agg = agg.reset_index()

# Some cleaning/formats for readability
agg['mean_helpful_votes'] = agg['mean_helpful_votes'].fillna(0).round(2)
agg['percent_verified'] = (agg['percent_verified'].fillna(0) * 100).round(1)
agg['positivity_rate'] = (agg['positivity_rate'].fillna(0) * 100).round(1)

# Save summary to CSV for later inspection
agg.to_csv('warranty_product_summary.csv', index=False)

print('Aggregated', len(agg), 'products mentioning warranty/guarantee. Saved to warranty_product_summary.csv')

Aggregated 11749 products mentioning warranty/guarantee. Saved to warranty_product_summary.csv


In [20]:
# Quick inspection: show top products by number of warranty-related reviews and worst/best mean rating
display_columns = ['asin','mean_overall_warranty','count_reviews','positive_reviews','negative_reviews','positivity_rate','mean_helpful_votes','percent_verified','avg_review_length_words']

print('Top 10 products by count of warranty-related reviews:')
display(agg.sort_values('count_reviews', ascending=False)[display_columns].head(10))

print('Top 10 products with lowest mean overall (warranty-related reviews):')
display(agg.sort_values('mean_overall_warranty', ascending=True)[display_columns].head(10))

print('Top 10 products with highest mean overall (warranty-related reviews):')
display(agg.sort_values('mean_overall_warranty', ascending=False)[display_columns].head(10))

# Show a couple example reviews for the single worst product (if exists) to inspect content
if len(agg):
    worst_asin = agg.sort_values('mean_overall_warranty', ascending=True).iloc[0]['asin']
    print('Example negative reviews mentioning warranty/guarantee for asin =', worst_asin)
    examples = train_data_filtered[train_data_filtered['asin'] == worst_asin].sort_values('unixReviewTime', ascending=False).head(5)
    display(examples[['overall','vote_num','verified_flag','reviewerName','summary','reviewText']])

Top 10 products by count of warranty-related reviews:


,asin,mean_overall_warranty,count_reviews,positive_reviews,negative_reviews,positivity_rate,mean_helpful_votes,percent_verified,avg_review_length_words
4732,B00LF10KTE,4.173913,46,37,9,80.4,1.59,82.6,140.586957
8699,B0163CHND0,1.577778,45,4,41,8.9,7.33,62.2,256.466667
4133,B00IVPU7AO,1.659091,44,6,38,13.6,21.00,90.9,137.136364
3063,B00DSUTX3O,4.159091,44,36,8,81.8,1.02,95.5,120.977273
2089,B008VQ8IKY,3.555556,36,23,13,63.9,2.86,91.7,137.833333
1227,B004LB5AZY,4.514286,35,32,3,91.4,1.49,51.4,244.857143
6051,B00S9SGNNS,2.968750,32,15,17,46.9,1.22,90.6,267.718750
7994,B013HSW4N2,3.032258,31,16,15,51.6,1.61,51.6,235.290323
5074,B00N1O1NQW,5.000000,29,29,0,100.0,6.79,75.9,87.724138
333,B000HPV3RW,3.964286,28,22,6,78.6,0.57,92.9,123.785714


Top 10 products with lowest mean overall (warranty-related reviews):


,asin,mean_overall_warranty,count_reviews,positive_reviews,negative_reviews,positivity_rate,mean_helpful_votes,percent_verified,avg_review_length_words
0,6541654530,1.0,1,0,1,0.0,0.00,100.0,104.0
2008,B00858I5OC,1.0,1,0,1,0.0,0.00,100.0,53.0
5908,B00RH69WRM,1.0,1,0,1,0.0,2.00,0.0,368.0
2010,B0085H18W4,1.0,3,0,3,0.0,5.67,66.7,358.0
5888,B00RCYEL7U,1.0,3,0,3,0.0,16.00,66.7,293.0
5885,B00RBQHC1G,1.0,1,0,1,0.0,0.00,100.0,42.0
5883,B00RBHGWRK,1.0,1,0,1,0.0,0.00,100.0,37.0
2017,B00877ZOYK,1.0,1,0,1,0.0,4.00,100.0,181.0
5878,B00RBG60ZK,1.0,1,0,1,0.0,2.00,100.0,201.0
5871,B00R98JVVU,1.0,1,0,1,0.0,0.00,100.0,122.0


Top 10 products with highest mean overall (warranty-related reviews):


,asin,mean_overall_warranty,count_reviews,positive_reviews,negative_reviews,positivity_rate,mean_helpful_votes,percent_verified,avg_review_length_words
11748,B01HJ8E11E,5.0,1,1,0,100.0,3.0,0.0,286.0
4659,B00L21Q5LY,5.0,1,1,0,100.0,17.0,100.0,80.0
9762,B01A5TNHWS,5.0,1,1,0,100.0,3.0,100.0,162.0
4655,B00L1O2PDY,5.0,1,1,0,100.0,6.0,100.0,1340.0
4654,B00L1LXOWS,5.0,1,1,0,100.0,3.0,0.0,244.0
4653,B00L1G6GCS,5.0,1,1,0,100.0,0.0,100.0,57.0
9763,B01A5UI5V0,5.0,2,2,0,100.0,0.0,0.0,128.5
9764,B01A60I4P6,5.0,2,2,0,100.0,27.0,0.0,3601.0
9767,B01A6B240Q,5.0,1,1,0,100.0,7.0,0.0,235.0
4645,B00L0IZZWY,5.0,1,1,0,100.0,0.0,100.0,55.0


Example negative reviews mentioning warranty/guarantee for asin = 6541654530


,overall,vote_num,verified_flag,reviewerName,summary,reviewText
734211,1,0.0,True,DannyPics,Lens doesn't work anymore,Originally the lens worked well. It hasn't bee...


In [21]:
# Post-processing: convert unix times to datetime and merge product titles/brands if available
import pandas as pd
merged = agg.copy()
# Convert unix timestamp columns (if present) to readable datetimes
for col in ['first_review_time_unix','last_review_time_unix']:
    if col in merged.columns:
        merged[col.replace('_unix','_date')] = pd.to_datetime(merged[col], unit='s', errors='coerce')

# Merge title/brand info if title_brand DataFrame is available and contains 'asin'
if 'title_brand' in globals():
    tb = title_brand.copy()
    if 'asin' in tb.columns:
        # pick common metadata columns if they exist
        meta_cols = [c for c in ['title','brand','product_title','product_name'] if c in tb.columns]
        use_cols = ['asin'] + meta_cols if meta_cols else ['asin']
        try:
            merged = merged.merge(tb[use_cols].drop_duplicates('asin'), on='asin', how='left')
        except Exception as e:
            print('Could not merge title_brand:', e)

# Save enhanced summary
merged.to_csv('warranty_product_summary_with_titles.csv', index=False)
print('Saved warranty_product_summary_with_titles.csv')
display(merged.head())

Saved warranty_product_summary_with_titles.csv


,asin,mean_overall_warranty,count_reviews,positive_reviews,negative_reviews,positivity_rate,mean_helpful_votes,percent_verified,avg_review_length_words,avg_summary_length_words,first_review_time_unix,last_review_time_unix,first_review_time_date,last_review_time_date,title,brand
0,6541654530,1.0,1,0,1,0.0,0.0,100.0,104.0,4.0,1475539200,1475539200,2016-10-04,2016-10-04,Canon EF 24-105mm f/4L IS USM Lens Bundle Inte...,Canon
1,9800466657,5.0,1,1,0,100.0,0.0,100.0,214.0,9.0,1492387200,1492387200,2017-04-17,2017-04-17,SanDisk EXTREME PRO 64GB (95MB/s) MicroSDXC wo...,SanDisk
2,B000001OM4,4.0,1,1,0,100.0,0.0,100.0,145.0,5.0,1470268800,1470268800,2016-08-04,2016-08-04,Maxell CD-330 CD-to-Cassette Audio Adapter (19...,Maxell
3,B000001OM5,5.0,1,1,0,100.0,0.0,100.0,118.0,13.0,1492473600,1492473600,2017-04-18,2017-04-18,Maxell Safe and Effective Feature CD Player an...,Maxell
4,B000001ON6,5.0,1,1,0,100.0,0.0,100.0,108.0,11.0,1500076800,1500076800,2017-07-15,2017-07-15,Maxell 290038 Vhs Cleaner Wet Premium,Maxell
